In [65]:
import pickle
import torch
from vllm.vllm_flash_attn import flash_attn_varlen_func

In [66]:
with open('/root/vllm/eval/debug/flash_attn_inputs.pkl', 'rb') as f:
    data = pickle.load(f)


FileNotFoundError: [Errno 2] No such file or directory: '/root/vllm/eval/debug/flash_attn_inputs.pkl'

In [67]:
q = data['q']
k = data['k']
v = data['v']
cu_seqlens_q = data['cu_seqlens_q']
cu_seqlens_k = data['cu_seqlens_k']
max_seqlen_q = data['max_seqlen_q']
max_seqlen_k = data['max_seqlen_k']
softmax_scale = data['softmax_scale']
causal = data['causal']
window_size = data['window_size']
alibi_slopes = data['alibi_slopes']
softcap = data['softcap']
block_table = data['block_table']
out = data['out']

print(block_table[:, :-10].shape)

torch.Size([50, 1250])


In [71]:
k = torch.zeros(8000, 16, 8, 128).cuda().to(torch.bfloat16)
v = torch.zeros(8000, 16, 8, 128).cuda().to(torch.bfloat16)

In [72]:
import time
times = []
flash_attn_varlen_func(
    q=q,
    k=k,
    v=v,
    cu_seqlens_q=cu_seqlens_q,
    cu_seqlens_k=cu_seqlens_k,
    max_seqlen_q=max_seqlen_q,
    max_seqlen_k=max_seqlen_k,
    softmax_scale=softmax_scale,
    causal=causal,
    window_size=window_size,
    alibi_slopes=alibi_slopes,
    block_table=block_table,
    out=out,
    )
torch.cuda.synchronize()
for _ in range(100):
    start = time.perf_counter()
    
    flash_attn_varlen_func(
    q=q,
    k=k,
    v=v,
    cu_seqlens_q=cu_seqlens_q,
    cu_seqlens_k=cu_seqlens_k,
    max_seqlen_q=max_seqlen_q,
    max_seqlen_k=max_seqlen_k,
    softmax_scale=softmax_scale,
    causal=causal,
    window_size=window_size,
    alibi_slopes=alibi_slopes,
    block_table=block_table,
    out=out,
    )
    
    torch.cuda.synchronize()
    end = time.perf_counter()
    times.append((end - start) * 1000)  # Convert to milliseconds

print(torch.tensor(times).mean())

tensor(9.6205)


In [73]:
import time

block_table[:, :-10] = torch.randint(8000, (1,1250,))
times = []
flash_attn_varlen_func(
    q=q,
    k=k,
    v=v,
    cu_seqlens_q=cu_seqlens_q,
    cu_seqlens_k=cu_seqlens_k,
    max_seqlen_q=max_seqlen_q,
    max_seqlen_k=max_seqlen_k,
    softmax_scale=softmax_scale,
    causal=causal,
    window_size=window_size,
    alibi_slopes=alibi_slopes,
    block_table=block_table,
    out=out,
    )
torch.cuda.synchronize()
for _ in range(100):
    start = time.perf_counter()
    
    flash_attn_varlen_func(
    q=q,
    k=k,
    v=v,
    cu_seqlens_q=cu_seqlens_q,
    cu_seqlens_k=cu_seqlens_k,
    max_seqlen_q=max_seqlen_q,
    max_seqlen_k=max_seqlen_k,
    softmax_scale=softmax_scale,
    causal=causal,
    window_size=window_size,
    alibi_slopes=alibi_slopes,
    block_table=block_table,
    out=out,
    )
    
    torch.cuda.synchronize()
    end = time.perf_counter()
    times.append((end - start) * 1000)  # Convert to milliseconds

print(torch.tensor(times).mean())

tensor(9.6595)


In [51]:
torch.randint(4000, (10,))

tensor([ 142, 1876,  728,  734, 3608,  661, 3032, 2848, 3119, 2931])